# Building an E-commerce Recommendation Engine with Kumo AI

In this notebook we'll build an ecommerce recommendation engine using Kumo AI with **G**raph **N**eural **N**etworks (GNNs). By the end, we'll have a system that can:

- **Predict customer lifetime value** - Identify our most valuable customers for the next 30 days
- **Generate personalized product recommendations** - Show customers exactly what they're likely to buy (great for chatbot-recommendations)
- **Forecast purchase behavior** - Know which customers are about to make purchases

All using real-world H&M retail data (with 33M+ transactions).

## GNNs and Recommendation Engines

Before diving in, let's quickly cover _why_ we're using GNNs for recommendations.

Traditional recommendation systems (RecSys) often miss complex relationships - in our case that would be the _important_ relationships between customers, products, and transactions. GNNs excel at capturing these relationships, allowing us to understand:

- **Network Effects**: GNNs naturally model how customer preferences influence each other
- **Temporal Dynamics**: They understand how purchasing patterns evolve over time
- **Cold Start Problem**: GNNs can make predictions for new customers based on similar network patterns

Kumo AI helps us skip the hard part of curating data and training a GNN. Instead Kumo allows us to turn up with our data and a set of predictions we'd like to make, and Kumo auto-trains GNNs to get those predictions. Not only that, but Kumo AI has been built by experts on GNNs, meaning we're getting top-tier performance that would be impossible to achieve without deep expertise in graph theory or neural nets.

## Setting Up Your Environment

First, let's install the necessary packages. We'll need:

- `kumoai` - The main Kumo AI SDK for building and training GNN models
- `kaggle` — The source for the H&M ecommerce dataset we'll be using
- `db-dtypes` - Support for BigQuery data types when working with our data warehouse
- `google-auth` and `google-cloud-bigquery` — to get access to our source data in BigQuery

In [1]:
!pip install -qU \
    "db-dtypes>=1.4.3" \
    "google-auth>=2.40.2" \
    "google-cloud-bigquery>=3.33.0" \
    "ipykernel>=6.29.5" \
    "kaggle>=1.7.4.5" \
    "kumoai>=2.1.0" \
    "relbench>=1.1.0"

## Connecting to Kumo


Kumo AI provides a cloud-based platform for training and deploying Graph Neural Networks. To get started, you'll need an API key from your Kumo workspace.

![Screenshot of the Kumo AI dashboard showing the API keys section, with the interface displaying where users can create and manage their API keys](https://github.com/aurelio-labs/cookbook/blob/main/recsys/ecommerce/kumo-hm/assets/kumo-ui-api-key.png)

In [1]:
import os
from getpass import getpass
import kumoai

api_key = os.getenv("KUMO_API_KEY") or \
    getpass("Enter your Kumo API key: ")

kumoai.init(url="https://aurelio.kumoai.cloud/api", api_key=api_key)

/Users/jamesbriggs/Documents/aurelio/kumo-advocacy/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[2025-07-11 15:34:56 - kumoai:196 - INFO] Successfully initialized the Kumo SDK against deployment https://aurelio.kumoai.cloud/api, with log level INFO.


## Data Connectors


Kumo seamlessly integrates with your existing data infrastructure. While it supports various data storage providers including S3, Snowflake, and Databricks, we'll focus on BigQuery for this tutorial.

### Setting Up BigQuery Permissions

Before connecting Kumo to BigQuery, you'll need to create a service account in GCP with the following permissions:

* BigQuery Data Viewer
* BigQuery Filtered Data Viewer
* BigQuery Metadata Viewer
* BigQuery Read Session User
* BigQuery User
* BigQuery Data Editor

Once you have created a service account with these permissions, go to **Manage keys** > **Add key** > choose **Key type: JSON** > download the credentials file. Store it in the same directory as this code and rename it to `kumo-gcp-creds.json`.

![Diagram showing the flow from BigQuery to Kumo: we get BigQuery service account JSON from GCP → Kumo connector → read/write access between Kumo and BigQuery](https://github.com/aurelio-labs/cookbook/blob/main/recsys/ecommerce/kumo-hm/assets/kumo-bigquery-connection.png)


### Creating Your BigQuery Connector


Now let's establish the connection between Kumo and BigQuery. This connector will be our pipeline for reading data from BigQuery and writing predictions back:

In [6]:
import kumoai
import json

name = "kumo_intro_live"  # call this whatever you like
project_id = "aurelio-advocacy"  # enter your GCP project ID
dataset_id = "rel_hm" # a unique ID for this dataset

with open("kumo-gcp-creds.json", "r") as fp:
    creds = json.loads(fp.read())

connector = kumoai.BigQueryConnector(
    name=name,
    project_id=project_id,
    dataset_id=dataset_id, 
    credentials=creds,
)

When we first try to view tables we'll see an error as we have not added any tables yet:

In [7]:
from kumoai.exceptions import HTTPException

try:
    connector.table_names()
except HTTPException as e:
    print(e)

Don't worry about this error - it's expected! We haven't added any tables to our BigQuery dataset yet. Let's fix that by loading the H&M data.

For this to work, we'll need some data.


## The H&M Dataset


We'll be working with H&M's actual transaction data - this isn't a toy dataset! With over 33 million transactions, 1.3 million customers, and 100,000+ products, this represents the kind of scale you'd encounter in production recommendation systems.

The dataset includes:
- **Customers**: Demographics and preferences of 1.3M shoppers
- **Articles**: Detailed information about 100K+ fashion products
- **Transactions**: 33M+ purchase records with timestamps

GNNs are perfect for developing an abstract but deep understanding of this type of large-scale interconnected data.

![Visualization showing the scale of the H&M dataset: 1.3M customers connected to 100K+ products through 33M+ transactions](https://github.com/aurelio-labs/cookbook/blob/main/recsys/ecommerce/kumo-hm/assets/h-and-m-dataset.png)


### Downloading from Kaggle


The H&M dataset is available through Kaggle's competition platform. We'll download it programmatically using the Kaggle API.

First, make sure you have:
1. A Kaggle account
2. Your API credentials downloaded from [kaggle.com/settings](https://www.kaggle.com/settings)
3. The credentials file saved to `~/.kaggle/kaggle.json`

You'll also need to accept the competition terms on the [H&M competition page](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations) by clicking "Late Submission".

In [9]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

To access the dataset you must first approve their T&Cs. You can do this on the competition page by clicking on "Late Submission".

In [ ]:
api.competition_download_files(
    competition="h-and-m-personalized-fashion-recommendations",
    quiet=False
)

Note that the dataset can take a _long_ time download — we need to patient here. Once the download is complete we can extract the file like so:

In [7]:
import zipfile

path = "h-and-m-personalized-fashion-recommendations.zip"

with zipfile.ZipFile(path, "r") as zip_ref:
    file_list = zip_ref.namelist()
    # we only want the csv files
    file_list = [f for f in file_list if f.endswith(".csv")]
    for file in file_list:
        zip_ref.extract(file, "hm_data")

We only need the `csv` files:

In [10]:
import pathlib

files = [str(x) for x in pathlib.Path("hm_data").glob("*.csv")]
files

['hm_data/customers.csv',
 'hm_data/articles.csv',
 'hm_data/transactions_train.csv',
 'hm_data/sample_submission.csv']


### Loading Data into BigQuery


Now comes the crucial step - loading our data into BigQuery where Kumo can access it. We'll create separate tables for customers, articles (products), and transactions.

We first authenticate ourselves:

In [11]:
from google.cloud import bigquery
from google.oauth2 import service_account

creds_filepath = "kumo-gcp-creds.json"

creds_obj = service_account.Credentials.from_service_account_file(
    creds_filepath,
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)

client = bigquery.Client(
    credentials=creds_obj,
    project="aurelio-advocacy",
)    

BigQuery is organized into a _dataset_ which can contain various tables — in our case one table is equivalent to a single CSV, of which we have three; `customers.csv`, `articles.csv`, and `transactions_train.csv`.

Let's create our `H&M Dataset` in BigQuery:

In [12]:
dataset_ref = client.dataset(dataset_id)

# create the dataset if it doesn't exist
try:
    dataset = client.get_dataset(dataset_ref)
    print("Dataset already exists")
except Exception:
    # if the dataset doesn't exist, create it
    print("Creating dataset...")
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = "US"  # as preferred
    dataset.description = "H&M Dataset"
    dataset = client.create_dataset(dataset)

Dataset already exists


Now we push each of our CSVs to their own tables in our new dataset:

In [13]:
files[:-1]

['hm_data/customers.csv',
 'hm_data/articles.csv',
 'hm_data/transactions_train.csv']

In [6]:
for file in files[:-1]:
    table_id = file.split("/")[-1].split(".")[0]
    print(f"Pushing {table_id} to BigQuery...")
    table_ref = dataset.table(table_id)
    # setup the load job
    job_config = bigquery.LoadJobConfig(
        # overwrite the table if it already exists
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        source_format=bigquery.SourceFormat.CSV,
        # skip headers (BQ reads these separately)
        skip_leading_rows=1,
        # autodetect the schema
        autodetect=True,
    )
    with open(file, "rb") as f:
        # load table from file into BQ
        load_job = client.load_table_from_file(
            f, table_ref, job_config=job_config
        )
    load_job.result()  # wait for the job to complete
    # print the job results
    print(f"Loaded {load_job.output_rows} rows")


## Building Your Graph with Kumo


With our data in BigQuery, we can now construct the graph structure that Kumo will use for training. This involves:

1. **Defining tables** - Tell Kumo about our data schema
2. **Setting relationships** - Connect customers to transactions to products
3. **Creating the graph** - Build the complete network structure

![Diagram showing the graph structure: Customer nodes connect to Transaction nodes, which connect to Article nodes](https://github.com/aurelio-labs/cookbook/blob/main/recsys/ecommerce/kumo-hm/assets/graph-connections.png)


### Understanding Your Data Schema


Before building our graph, let's examine the structure of each table. This helps us identify:
- Primary keys for entity identification
- Foreign keys for relationships
- Temporal columns for time-based predictions

In [14]:
# view available tables
connector.table_names()

['transactions_test',
 'sum_transactions_pred_predictions',
 'customers',
 'articles',
 'PURCHASE_PRED_predictions',
 'TRANSACTIONS_PRED_predictions',
 'SUM_TRANSACTIONS_PRED_predictions',
 'transactions_train']

We connect to each of the _source_ tables like so:

In [15]:
articles_source = connector["articles"]
customers_source = connector["customers"]
transactions_source = connector["transactions_train"]

We can view these source tables like so:

In [18]:
articles_source.columns

[SourceColumn(name='article_id', stype=ID, dtype=int64, is_primary=False),
 SourceColumn(name='product_code', stype=ID, dtype=int64, is_primary=False),
 SourceColumn(name='prod_name', stype=categorical, dtype=string, is_primary=False),
 SourceColumn(name='product_type_no', stype=ID, dtype=int64, is_primary=False),
 SourceColumn(name='product_type_name', stype=categorical, dtype=string, is_primary=False),
 SourceColumn(name='product_group_name', stype=categorical, dtype=string, is_primary=False),
 SourceColumn(name='graphical_appearance_no', stype=ID, dtype=int64, is_primary=False),
 SourceColumn(name='graphical_appearance_name', stype=categorical, dtype=string, is_primary=False),
 SourceColumn(name='colour_group_code', stype=ID, dtype=int64, is_primary=False),
 SourceColumn(name='colour_group_name', stype=categorical, dtype=string, is_primary=False),
 SourceColumn(name='perceived_colour_value_id', stype=ID, dtype=int64, is_primary=False),
 SourceColumn(name='perceived_colour_value_name

In [16]:
articles_source.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,523776009,523776,NAOMI PADDED JACKET,262,Jacket,Garment Upper body,1010016,Solid,41,Light Red,...,Outwear,A,Ladieswear,1,Ladieswear,19,Womens Jackets,1007,Outdoor,"Padded jacket with a high stand-up collar, zip..."
1,523915001,523915,Jessica sporty jacket,264,Blazer,Garment Upper body,1010016,Solid,40,Other Red,...,Outwear,A,Ladieswear,1,Ladieswear,19,Womens Jackets,1007,Outdoor,Jacket in a cotton-blend weave with a hood wit...
2,535716001,535716,Pearl padded v5,262,Jacket,Garment Upper body,1010016,Solid,1,Other,...,Outwear,A,Ladieswear,1,Ladieswear,19,Womens Jackets,1007,Outdoor,"Long, padded jacket in a quilted weave with a ..."
3,536524006,536524,Temple,262,Jacket,Garment Upper body,1010016,Solid,63,Dark Purple,...,Outwear,A,Ladieswear,1,Ladieswear,19,Womens Jackets,1007,Outdoor,"Padded jacket with a stand-up collar, zip with..."
4,549607002,549607,Annie fur,262,Jacket,Garment Upper body,1010005,Colour blocking,70,Other Blue,...,Outwear,A,Ladieswear,1,Ladieswear,19,Womens Jackets,1007,Outdoor,"Jacket in soft faux fur with a collar, conceal..."


From this, we know our `article_id` should be used as the primary key for the **articles** table. We need to perform the same check for **customers** and **transactions**:

In [22]:
customers_source.head(2)

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,000064249685c11552da43ef22a5030f35a147f723d5b0...,NaN,NaN,NaN,NaN,NaN,2c29ae653a9282cce4151bd87643c907644e09541abc28...
1,004d89470677ce579a313b2259e93b07ea6f944686f9c3...,NaN,NaN,NaN,NaN,NaN,2c29ae653a9282cce4151bd87643c907644e09541abc28...


In [23]:
transactions_source.head(2)  # no primary key

,t_dat,customer_id,article_id,price,sales_channel_id
0,2019-04-28,74133dd63c40245a9bd1b95d1657598e8763347c0d13db...,499807002,0.00016949153,1
1,2020-05-27,4dcc9a17cc53620d442869b19b371deb2a4b3dc34d19f6...,523488001,0.00020338983,1


For **customers** the primary ID is `customer_id`. For **transactions** there isn't a primary key, so we will avoid setting that value below. However, we do have a date column in **transactions** - we'll use that in a moment.

We'll use the primary keys and date column identified above when defining our Kumo-version of these tables below. To create our Kumo tables we use the `kumo.Table.from_source_table` method. We identify `primary_key` columns and any `time_column`.


### Creating Kumo Table Objects


Now we transform our BigQuery tables into Kumo table objects. This step is crucial as it:
- Identifies primary keys for entity resolution
- Recognizes temporal columns for time-aware predictions
- Infers data types and metadata automatically

In [24]:
articles = kumoai.Table.from_source_table(
    source_table=articles_source,
    primary_key="article_id"
).infer_metadata()

customers = kumoai.Table.from_source_table(
    source_table=customers_source,
    primary_key="customer_id"
).infer_metadata()

transactions_train = kumoai.Table.from_source_table(
    source_table=transactions_source,
    time_column="t_dat"
).infer_metadata()

We can view our table's inferred metadatas like so:

In [25]:
articles.metadata

,name,dtype,stype,is_primary_key,is_time_column,is_end_time_column
0,article_id,int64,ID,True,False,False
1,product_code,int64,ID,False,False,False
2,prod_name,string,categorical,False,False,False
3,product_type_no,int64,ID,False,False,False
4,product_type_name,string,categorical,False,False,False
5,product_group_name,string,categorical,False,False,False
6,graphical_appearance_no,int64,ID,False,False,False
7,graphical_appearance_name,string,categorical,False,False,False
8,colour_group_code,int64,ID,False,False,False
9,colour_group_name,string,categorical,False,False,False



### Defining Your Graph Structure


The final step is connecting everything into a graph. We define edges that represent the relationships between our tables:

In [26]:
graph = kumoai.Graph(
    # tables that will be in the graph, key is the table name and value is the table object
    tables={
        "articles": articles,
        "customers": customers,
        "transactions": transactions_train,
    },
    # then define primary key <> foreign key relationships
    edges=[
        {"src_table": "transactions", "fkey": "customer_id", "dst_table": "customers"},
        {"src_table": "transactions", "fkey": "article_id", "dst_table": "articles"},
    ]
)

# validate the graph
graph.validate(verbose=True)

[2025-07-11 16:05:42 - kumoai.graph.table:555 - INFO] Table articles is configured correctly.
[2025-07-11 16:05:44 - kumoai.graph.table:555 - INFO] Table customers is configured correctly.
[2025-07-11 16:05:45 - kumoai.graph.table:555 - INFO] Table transactions_train is configured correctly.
[2025-07-11 16:05:47 - kumoai.graph.graph:798 - INFO] Graph is configured correctly.


Graph(
  tables=[articles, customers, transactions],
  edges=[Edge(src_table='transactions', fkey='customer_id', dst_table='customers'), Edge(src_table='transactions', fkey='article_id', dst_table='articles')],
)

We can visualize our graph with `graph.visualize()`, this does require that we have the `graphviz` python package installed _and_ graphviz executables, which are [platform-specific](https://graphviz.org/download/). Alternatively, we can also find our graph in the Kumo UI.

## Predictive Query Language (PQL)

Here's where Kumo really shines. Instead of writing complex neural network code, you describe what you want to predict using **P**redictive **Q**uery **L**anguage (PQL) - a SQL-like language designed for predictive tasks.

PQL allows you to express sophisticated predictive queries in an intuitive way. Let's explore three powerful use cases for our recommendation system.


### Use Case 1: Predicting Customer Value


Our first prediction will identify high-value customers. We'll predict the total revenue each customer will generate over the next 30 days.

![PQL syntax diagram for customer value prediction showing: PREDICT SUM(transactions.price, 0, 30) FOR EACH customers.customer_id with annotations explaining each component](https://github.com/aurelio-labs/cookbook/blob/main/recsys/ecommerce/kumo-hm/assets/pql-sum-price.png)

This PQL statement contains several key components:

* **Target**: What we're predicting - the `SUM` of `transactions.price`
* **Time Window**: Next 30 days (from day 0 to day 30)
* **Entity**: Who we're predicting for - each `customer_id`

This helps us focus marketing efforts on customers most likely to make significant purchases.

In [27]:
pquery = kumoai.PredictiveQuery(
    graph=graph,
    query=(
        "PREDICT SUM(transactions.price, 0, 30, days)\n"
        "FOR EACH customers.customer_id\n"
    )
)

# validate our pquery
pquery.validate(verbose=True)

[2025-07-11 16:12:47 - kumoai.pquery.predictive_query:211 - INFO] Query PREDICT SUM(transactions.price, 0, 30, days)
FOR EACH customers.customer_id
 is configured correctly.


Now we can train a GNN on this pquery. Kumo will automatically identify the likely best GNN parameters for us using the `suggest_model_plan` method:

In [28]:
model_plan = pquery.suggest_model_plan()
model_plan

ModelPlan(
  training_job=TrainingJobPlan(
    num_experiments=4,
    metrics=['mae', 'mse', 'rmse'],
    tune_metric='mae',
    pruning=Pruning(min_epochs=2, k=3, min_delta=0.0, patience=1),
    refit_trainval=False,
    refit_full=False,
  ),
  column_processing=ColumnProcessingPlan(
    encoder_overrides=None,
  ),
  neighbor_sampling=NeighborSamplingPlan(
    num_neighbors=[
      NumNeighbors(
        hop1={
          default=12,
          customers.customer_id->transactions.customer_id=inferred,
        },
        hop2=12,
      ),
    ],
  ),
  optimization=OptimizationPlan(
    max_epochs=8,
    min_steps_per_epoch=30,
    max_steps_per_epoch=1000,
    max_val_steps=1000,
    max_test_steps=2000,
    loss=[
      huber,
    ],
    base_lr=[
      0.0001,
      0.0005,
      0.001,
      0.005,
      0.01,
      0.05,
    ],
    weight_decay=[
      0.0,
      5e-08,
      5e-07,
      5e-06,
    ],
    batch_size=[
      512,
      1024,
    ],
    early_stopping=[
      EarlyS

Training will run asynchronously in Kumo's cloud infrastructure. You can monitor progress through the status method or in the Kumo UI.

If we're happy with Kumo's recommended parameters, we train like so:

In [29]:
trainer = kumoai.Trainer(model_plan=model_plan)
training_job = trainer.fit(
    graph=graph,
    train_table=pquery.generate_training_table(non_blocking=True),
    non_blocking=True,
)

[2025-07-11 16:17:57 - kumoai.graph.graph:394 - INFO] Graph snapshot with identifier graph-f0e424cbb38b21d712085591d0592c35@5f744dd5f9ccd70ad24ba4bb5d08962b created.
[2025-07-11 16:18:00 - kumoai.graph.graph:462 - WARNING] Graph snapshot with identifier graph-f0e424cbb38b21d712085591d0592c35@5f744dd5f9ccd70ad24ba4bb5d08962b already exists, and will not be refreshed.


Training will take some time, but we can check in on our training job progress with `training_job.status()`:

In [30]:
training_job.status()

JobStatusReport(status=RUNNING, tracking_url='https://aurelio.kumoai.cloud/jobs/training/trainingjob-dc1da7da14e24303a64dbb4c38478102', start_time=datetime.datetime(2025, 7, 11, 14, 18, 0, 665322, tzinfo=datetime.timezone.utc), end_time=None, event_log=[JobEventLogEntry(stage_name='Ingesting Data', last_updated_at=datetime.datetime(2025, 7, 10, 20, 10, 10, 31150, tzinfo=datetime.timezone.utc), detail='This job is processing data from your warehouse.')], validation_response=None)

### Use Case 2: Personalized Product Recommendations



While our first model trains, let's set up a more sophisticated prediction - personalized product recommendations. This query predicts the top 10 products each customer is most likely to purchase:

![PQL syntax diagram showing: PREDICT LIST_DISTINCT(transactions.article_id, 0, 30) RANK TOP 10 FOR EACH customers.customer_id with component annotations](https://github.com/aurelio-labs/cookbook/blob/main/recsys/ecommerce/kumo-hm/assets/pql-purchases.png)

Key differences from our previous query:

* **LIST_DISTINCT**: Returns a list of unique products (no duplicates)
* **RANK TOP 10**: Filters to the 10 most likely purchases
* **Same time window**: Looking 30 days into the future

This powers "Recommended for You" sections in e-commerce applications.

In [31]:
purchase_pquery = kumoai.PredictiveQuery(
    graph=graph,
    query=(
        "PREDICT LIST_DISTINCT(transactions.article_id, 0, 30)\n"
        "RANK TOP 10\n"
        "FOR EACH customers.customer_id\n"
    )
)

# validate our pquery
purchase_pquery.validate(verbose=True)

[2025-07-11 16:23:59 - kumoai.pquery.predictive_query:211 - INFO] Query PREDICT LIST_DISTINCT(transactions.article_id, 0, 30)
RANK TOP 10
FOR EACH customers.customer_id
 is configured correctly.


As before, we get a model plan from Kumo and train with it:

In [32]:
model_plan = purchase_pquery.suggest_model_plan()
# start training
purchase_trainer = kumoai.Trainer(model_plan=model_plan)
purchase_training_job = purchase_trainer.fit(
    graph=graph,
    train_table=purchase_pquery.generate_training_table(non_blocking=True),
    non_blocking=True,
)

[2025-07-11 16:24:51 - kumoai.graph.graph:462 - WARNING] Graph snapshot with identifier graph-f0e424cbb38b21d712085591d0592c35@5f744dd5f9ccd70ad24ba4bb5d08962b already exists, and will not be refreshed.
[2025-07-11 16:24:53 - kumoai.graph.graph:462 - WARNING] Graph snapshot with identifier graph-f0e424cbb38b21d712085591d0592c35@5f744dd5f9ccd70ad24ba4bb5d08962b already exists, and will not be refreshed.


We'll check in on the job status occasionally with:

In [33]:
purchase_training_job.status()

JobStatusReport(status=RUNNING, tracking_url='https://aurelio.kumoai.cloud/jobs/training/trainingjob-bd643fd1284748f1843d22d0f3674447', start_time=datetime.datetime(2025, 7, 11, 14, 24, 54, 343401, tzinfo=datetime.timezone.utc), end_time=None, event_log=[JobEventLogEntry(stage_name='Ingesting Data', last_updated_at=datetime.datetime(2025, 7, 10, 20, 10, 10, 31150, tzinfo=datetime.timezone.utc), detail='This job is processing data from your warehouse.')], validation_response=None)


### Use Case 3: Predicting Purchase Volume


Our final prediction focuses on customer engagement. We'll predict how many purchases customers will make, but only for those who've been recently active:

![PQL syntax diagram: PREDICT COUNT(transactions.*, 0, 30) FOR EACH customers.customer_id WHERE COUNT(transactions.*, -30, 0) > 0 with annotations](https://github.com/aurelio-labs/cookbook/blob/main/recsys/ecommerce/kumo-hm/assets/pql-transactions.png)

This query introduces conditional filtering:

* **COUNT(transactions.*, 0, 30)**: Count all future transactions
* **WHERE clause**: Only include customers with recent activity
* **Negative time window (-30, 0)**: Looks at past 30 days

This helps identify which active customers might be churning or increasing their engagement.

In [34]:
transactions_pquery = kumoai.PredictiveQuery(
    graph=graph,
    query=(
        "PREDICT COUNT(transactions.*, 0, 30)\n"
        "FOR EACH customers.customer_id\n"
        "WHERE COUNT(transactions.*, -30, 0) > 0\n"
    )
)

# validate our pquery
transactions_pquery.validate(verbose=True)

[2025-07-11 16:27:30 - kumoai.pquery.predictive_query:211 - INFO] Query PREDICT COUNT(transactions.*, 0, 30)
FOR EACH customers.customer_id
WHERE COUNT(transactions.*, -30, 0) > 0
 is configured correctly.


In [35]:
model_plan = transactions_pquery.suggest_model_plan()
# start training
transactions_trainer = kumoai.Trainer(model_plan=model_plan)
transactions_training_job = transactions_trainer.fit(
    graph=graph,
    train_table=transactions_pquery.generate_training_table(non_blocking=True),
    non_blocking=True,
)

[2025-07-11 16:28:15 - kumoai.graph.graph:462 - WARNING] Graph snapshot with identifier graph-f0e424cbb38b21d712085591d0592c35@5f744dd5f9ccd70ad24ba4bb5d08962b already exists, and will not be refreshed.
[2025-07-11 16:28:18 - kumoai.graph.graph:462 - WARNING] Graph snapshot with identifier graph-f0e424cbb38b21d712085591d0592c35@5f744dd5f9ccd70ad24ba4bb5d08962b already exists, and will not be refreshed.


In [36]:
transactions_training_job.status()

JobStatusReport(status=QUEUED, tracking_url='https://aurelio.kumoai.cloud/jobs/training/trainingjob-ca3c37a14cc2414e9f5bfa5ed973f494', start_time=datetime.datetime(2025, 7, 11, 14, 28, 19, 164675, tzinfo=datetime.timezone.utc), end_time=None, event_log=[JobEventLogEntry(stage_name='Not Started', last_updated_at=datetime.datetime(2025, 7, 11, 14, 28, 19, 164675, tzinfo=datetime.timezone.utc), detail='This job has been queued and is waiting to start processing.')], validation_response=None)


## Making Predictions

With our models trained, it's time to generate predictions. Kumo handles the complexity of distributed inference, allowing you to generate predictions at scale.


### Running Batch Predictions


Let's check that all our models have finished training:

In [41]:
training_job.status(), purchase_training_job.status(), transactions_training_job.status()

(JobStatusReport(status=DONE, tracking_url='https://aurelio.kumoai.cloud/jobs/training/trainingjob-dc1da7da14e24303a64dbb4c38478102', start_time=datetime.datetime(2025, 7, 11, 14, 18, 0, 665322, tzinfo=datetime.timezone.utc), end_time=datetime.datetime(2025, 7, 11, 15, 37, 59, 679336, tzinfo=datetime.timezone.utc), event_log=[JobEventLogEntry(stage_name='Done', last_updated_at=datetime.datetime(2025, 7, 10, 20, 10, 10, 31196, tzinfo=datetime.timezone.utc), detail='This job successfully completed!')], validation_response=None),
 JobStatusReport(status=DONE, tracking_url='https://aurelio.kumoai.cloud/jobs/training/trainingjob-bd643fd1284748f1843d22d0f3674447', start_time=datetime.datetime(2025, 7, 11, 14, 24, 54, 343401, tzinfo=datetime.timezone.utc), end_time=datetime.datetime(2025, 7, 11, 15, 53, 44, 83095, tzinfo=datetime.timezone.utc), event_log=[JobEventLogEntry(stage_name='Done', last_updated_at=datetime.datetime(2025, 7, 10, 20, 10, 10, 31196, tzinfo=datetime.timezone.utc), detail

Once our PQuery models have been trained we can use them to make predictive queries. First, let's check in on their status:

In [42]:
from kumoai.artifact_export.config import OutputConfig

assert training_job.status().status == "DONE", f"Job status is {training_job.status().status}"

predictions = trainer.predict(
    graph=graph,
    prediction_table=pquery.generate_prediction_table(non_blocking=True),
    output_config=OutputConfig(
        output_types={"predictions"},  
        output_connector=connector,
        output_table_name="SUM_TRANSACTIONS_PRED",
    ),
    training_job_id=training_job.id,
    non_blocking=True,
)

[2025-07-11 18:12:47 - kumoai.graph.graph:462 - WARNING] Graph snapshot with identifier graph-f0e424cbb38b21d712085591d0592c35@5f744dd5f9ccd70ad24ba4bb5d08962b already exists, and will not be refreshed.
[2025-07-11 18:12:48 - kumoai.graph.graph:462 - WARNING] Graph snapshot with identifier graph-f0e424cbb38b21d712085591d0592c35@5f744dd5f9ccd70ad24ba4bb5d08962b already exists, and will not be refreshed.
[2025-07-11 18:12:51 - kumoai.trainer.trainer:418 - WARNING] Prediction produced the following warnings: Warnings:
1. For the optimal experience, it is recommended for output tables to only contain uppercase characters, numbers, and underscores


When making predictions that involve ranking, ie predictions where we can have a varying number of predictions _per entity_ - we must specify how many predictions we'd like for each entity via the `num_classes_to_return` parameter.

In [43]:
assert purchase_training_job.status().status == "DONE", f"Job status is {purchase_training_job.status().status}"

purchase_predictions = purchase_trainer.predict(
    graph=graph,
    prediction_table=purchase_pquery.generate_prediction_table(non_blocking=True),
    num_classes_to_return=10,  # specific to ranking tasks
    output_config=OutputConfig(
        output_types={"predictions"},
        output_connector=connector,
        output_table_name="PURCHASE_PRED",
    ),
    training_job_id=purchase_training_job.id,
    non_blocking=True,
)

[2025-07-11 18:17:38 - kumoai.graph.graph:462 - WARNING] Graph snapshot with identifier graph-f0e424cbb38b21d712085591d0592c35@5f744dd5f9ccd70ad24ba4bb5d08962b already exists, and will not be refreshed.
[2025-07-11 18:17:40 - kumoai.graph.graph:462 - WARNING] Graph snapshot with identifier graph-f0e424cbb38b21d712085591d0592c35@5f744dd5f9ccd70ad24ba4bb5d08962b already exists, and will not be refreshed.
[2025-07-11 18:17:42 - kumoai.trainer.trainer:418 - WARNING] Prediction produced the following warnings: Warnings:
1. For the optimal experience, it is recommended for output tables to only contain uppercase characters, numbers, and underscores


Our transaction predictions _don't_ require this:

In [44]:
assert transactions_training_job.status().status == "DONE", f"Job status is {transactions_training_job.status().status}"

transactions_predictions = transactions_trainer.predict(
    graph=graph,
    prediction_table=transactions_pquery.generate_prediction_table(non_blocking=True),
    output_config=OutputConfig(
        output_types={"predictions"},
        output_connector=connector,
        output_table_name="TRANSACTIONS_PRED",
    ),
    training_job_id=transactions_training_job.id,
    non_blocking=True,
)

[2025-07-11 18:18:24 - kumoai.graph.graph:462 - WARNING] Graph snapshot with identifier graph-f0e424cbb38b21d712085591d0592c35@5f744dd5f9ccd70ad24ba4bb5d08962b already exists, and will not be refreshed.
[2025-07-11 18:18:26 - kumoai.graph.graph:462 - WARNING] Graph snapshot with identifier graph-f0e424cbb38b21d712085591d0592c35@5f744dd5f9ccd70ad24ba4bb5d08962b already exists, and will not be refreshed.
[2025-07-11 18:18:28 - kumoai.trainer.trainer:418 - WARNING] Prediction produced the following warnings: Warnings:
1. For the optimal experience, it is recommended for output tables to only contain uppercase characters, numbers, and underscores


These predictions do take some time to run, which we can check the status of over in our Kumo workspace under the **Predictions** tab. Once predictions _are_ complete, we can view the results either in BigQuery, or by loading the source tables via our `connector`. By default, Kumo will append `_predictions` to the `output_table_name` that we provided - we can also confirm the output table from inside the Kumo UI.


### Understanding the Results


Kumo writes predictions back to BigQuery in dedicated tables. Let's examine what each prediction tells us:

**Customer Value Predictions** - Who will generate the most revenue?

In [45]:
sum_transactions_pred = connector["SUM_TRANSACTIONS_PRED_predictions"]
sum_transactions_pred.head(5)

,ENTITY,TARGET_PRED
0,87116a80de24725fcc2e940792672b7ebc378570baed12...,-0.028264934
1,1ee2844bbde1a2f67e68c9455f7543cc7ffbe5abb8b2d9...,-0.01820541
2,b9300bd241818271d0b1aec5c77e920f3e189e92c11dd8...,-0.01773219
3,7a1522baa5ad01953ff493d251e43943cb100a1fcba9c5...,-0.01577353
4,000a5f3c8be9167cb0d09dd8a17b6b54998e9e83faaf52...,-0.012955962


Note that these predictions seem very small, what we're seeing here is some of the smallest predictions for sum of transactions. To view the top values, we can use BigQuery directly, like so:

In [46]:
query = f"""
SELECT * FROM {dataset_id}.SUM_TRANSACTIONS_PRED_predictions
ORDER BY TARGET_PRED DESC
LIMIT 5
"""

client.query(query).to_dataframe()

/Users/jamesbriggs/Documents/aurelio/kumo-advocacy/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,ENTITY,TARGET_PRED
0,63d4ee9c373b7ec52fd03b319faf53f3f1f24763d8a3ac...,0.668505
1,f69cf6fca69045a8259f9554e318e00fbf5e8e758e88b1...,0.657948
2,be96311f48cf1049e0da065ab322fada512ee88486c371...,0.647882
3,203785d96661d87a84718e998664c1169f43aa21b677a1...,0.643395
4,17d6270f6f81ad1f7e5a1cb7ed8edb54bc00d0d5c2cde6...,0.640910


**Product Recommendations** - What will they buy?

In [47]:
purchase_predictions = connector["PURCHASE_PRED_predictions"]
purchase_predictions.head()

,ENTITY,CLASS,SCORE
0,d6c6a21ea841ed2f57e7e6b3f97e0393e7dcbd1840174e...,108775015,6.55312
1,c083fcd66b7bd180820191fc4df7efc8051fb29eeacc4a...,108775044,6.0066395
2,b8db7b733c30abf4303c9e46f904539aa87c322db35105...,108775044,6.849747
3,8270dfed94715623caa7c4e191e81adff2501609f7f29b...,108775044,8.27623
4,d7e14b1c9edc9882f38e599b21994c1f92d0920459f149...,108775044,5.6090508


**Transaction Volume** - How active will they be?

In [48]:
transactions_predictions = connector["TRANSACTIONS_PRED_predictions"]
transactions_predictions.head()

,ENTITY,TARGET_PRED
0,b1e954663f0f390287d69dc370f2692ff16bae59b0cc4a...,-0.32812172
1,9c9259cf8ce8b2243e5a54b7790ca6d3b71f471493a94e...,-0.30792093
2,ead63e07ef10b1a1995d0854f35c0cabffd750ae74824c...,-0.28713274
3,8276cf251bdfedfcd1e3488ba6a97e3119c615758ca3ac...,-0.28584266
4,3d9763439bb45d8f9607123b059f1c75cee2c4fd5acce8...,-0.27159393


Again, to find the largest predictions we use BigQuery:

In [49]:
query = f"""
SELECT * FROM {dataset_id}.TRANSACTIONS_PRED_predictions
ORDER BY TARGET_PRED DESC
LIMIT 5
"""

client.query(query).to_dataframe()

/Users/jamesbriggs/Documents/aurelio/kumo-advocacy/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,ENTITY,TARGET_PRED
0,2054e4750df99a84d020995d33d84a7031cb995026bf3c...,20.051224
1,8661ffd123c6b31c36c31e6377e34d3875ab3e73daedac...,19.587551
2,2cabdc6101018f8cea44310343769715049befed47caa9...,19.226614
3,ee69a199eaa2869700cf9750bfc2556f30e4f6d7a602ff...,19.197271
4,7ee82aa234ac383a2d687a98d5c75bc0e51f4cefebd843...,18.601965



## Bringing It All Together

Now the question is, what do we do with all this? We can:

* Identify next month's most valuable customers with `SUM_TRANSACTIONS_predictions`.
* See what those customers are most likely to buy with `PURCHASE_predictions`.
* Find how many items those customers are likely to buy with `TRANSACTIONS_predictions`.


### Finding Next Month's Most Valuable Customers

We'll be using BigQuery again for this step. To identify our most valuable customers we perform a `JOIN` on the `SUM_TRANSACTIONS_predictions.ENTITY` (filtered for the `TOP 30` scores) and `customers.customer_id` colums, giving us our top customers.

In [50]:
valuable_customers = f"""
SELECT cust.*, trans.target_pred AS score FROM {dataset_id}.customers cust
INNER JOIN (
    SELECT entity, target_pred FROM {dataset_id}.SUM_TRANSACTIONS_PRED_predictions
    ORDER BY target_pred DESC
    LIMIT 30
) trans ON cust.customer_id = trans.entity
"""

q = client.query(valuable_customers + ";")
# wait for query to complete
rows = q.result()

In [51]:
top_customers = rows.to_dataframe()
top_customers.head()

/Users/jamesbriggs/Documents/aurelio/kumo-advocacy/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code,score
0,e9c27cf3d00e7bb6a27f395a01d01fbe6328901afdb645...,NaN,NaN,ACTIVE,NONE,22,f59fe04dc1681c9aa47c962c570ca8125808658da62aba...,0.599846
1,ceb037bfdab35cdd507685b20648829ddc0d92c8e02e2f...,NaN,NaN,ACTIVE,NONE,24,e81510382de7956ee6d5a1fe7feedb49bed10d0bfb1121...,0.621152
2,d8c54f5ca6421ba8c5d7631ebdf7a5b67ccf2dce4b859c...,NaN,NaN,ACTIVE,NONE,25,3ef8732185461d61ebdd71c4ae4d4f7b148283e0633d8f...,0.585189
3,8c40103139dd4b93163fa25a536cac2351ebb5936700cb...,1.0,1.0,ACTIVE,Regularly,25,25bacc574b76b1cab001f5b6aa6df9caf6868c3dca7ff9...,0.575516
4,d1bbee89e5364ecdb031e2b2f4be3509029d007eac99a2...,1.0,1.0,ACTIVE,Regularly,37,e36ef41bd3c46633339136aac52842196a6cb257efc211...,0.610632


### What Will Those Customers Buy?

Now let's identify a customer's most likely purchases - an invaluable insight for personalized marketing. To get this data we'll be joining our customer to their highest probability purchases via the `PURCHASE_predictions` table like so:

* `PURCHASE_predictions.ENTITY` joins `customers.customer_id`
* `PURCHASE_predictions.CLASS` joins `articles.article_id`

Let's build the query:

In [52]:
product_recs = f"""
SELECT
    pred.entity AS customer_id,
    pred.score AS score,
    art.*
FROM {dataset_id}.PURCHASE_PRED_predictions pred
INNER JOIN (
    SELECT *
    FROM {dataset_id}.articles
) art ON pred.class = art.article_id
INNER JOIN (
    SELECT *
    FROM {dataset_id}.customers cust
    WHERE cust.customer_id = '{top_customers.customer_id[0]}'
) cust ON pred.entity = cust.customer_id;
"""

q = client.query(product_recs)
rows = q.result()

In [53]:
top_cust_recs = rows.to_dataframe()
top_cust_recs

/Users/jamesbriggs/Documents/aurelio/kumo-advocacy/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,customer_id,score,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,e9c27cf3d00e7bb6a27f395a01d01fbe6328901afdb645...,7.167438,787285001,787285,Magic,265,Dress,Garment Full body,1010001,All over pattern,...,Dress,A,Ladieswear,1,Ladieswear,15,Womens Everyday Collection,1013,Dresses Ladies,Calf-length dress in a patterned viscose weave...
1,e9c27cf3d00e7bb6a27f395a01d01fbe6328901afdb645...,7.023399,859957001,859957,LE Good Ada Dress,265,Dress,Garment Full body,1010016,Solid,...,Limited Edition,A,Ladieswear,1,Ladieswear,82,Special Collections,1023,Special Offers,"Long, wide dress in a cool viscose weave with ..."
2,e9c27cf3d00e7bb6a27f395a01d01fbe6328901afdb645...,6.783765,758381002,758381,Twist fancy,92,Heeled sandals,Shoes,1010016,Solid,...,Heels,C,Ladies Accessories,1,Ladieswear,64,Womens Shoes,1020,Shoes,Imitation leather sandals with a braided foot ...
3,e9c27cf3d00e7bb6a27f395a01d01fbe6328901afdb645...,6.890747,935635002,935635,LUCKY TIE NECK SHIRT,259,Shirt,Garment Upper body,1010001,All over pattern,...,Tops Woven,D,Divided,2,Divided,53,Divided Collection,1010,Blouses,"Blouse in woven fabric with a collar, ties and..."
4,e9c27cf3d00e7bb6a27f395a01d01fbe6328901afdb645...,6.762956,787285003,787285,Magic,265,Dress,Garment Full body,1010001,All over pattern,...,Dress,A,Ladieswear,1,Ladieswear,15,Womens Everyday Collection,1013,Dresses Ladies,Calf-length dress in a patterned viscose weave...
5,e9c27cf3d00e7bb6a27f395a01d01fbe6328901afdb645...,7.849483,904625001,904625,Pax HW PU Joggers,272,Trousers,Garment Lower body,1010016,Solid,...,Trousers,D,Divided,2,Divided,53,Divided Collection,1009,Trousers,Joggers in imitation leather. High waist with ...
6,e9c27cf3d00e7bb6a27f395a01d01fbe6328901afdb645...,7.308242,918212001,918212,ED Uma dress,265,Dress,Garment Full body,1010016,Solid,...,Woven top,A,Ladieswear,1,Ladieswear,2,H&M+,1010,Blouses,Calf-length dress in a cotton weave with a col...
7,e9c27cf3d00e7bb6a27f395a01d01fbe6328901afdb645...,7.370093,787285005,787285,Magic,265,Dress,Garment Full body,1010001,All over pattern,...,Dress,A,Ladieswear,1,Ladieswear,15,Womens Everyday Collection,1013,Dresses Ladies,Calf-length dress in a patterned viscose weave...
8,e9c27cf3d00e7bb6a27f395a01d01fbe6328901afdb645...,7.245024,814980001,814980,Alabama Dress,265,Dress,Garment Full body,1010011,Metallic,...,Dress,A,Ladieswear,1,Ladieswear,11,Womens Tailoring,1013,Dresses Ladies,Calf-length dress woven in an airy Tencel™ lyo...
9,e9c27cf3d00e7bb6a27f395a01d01fbe6328901afdb645...,6.791434,835247001,835247,Supernova,265,Dress,Garment Full body,1010004,Check,...,Dress,A,Ladieswear,1,Ladieswear,15,Womens Everyday Collection,1013,Dresses Ladies,Calf-length dress in a softly draping viscose ...


Those all look consistent - this specific customer clearly likes dresses.

#### Purchase Volume

Now let's now try the final prediction - how much volume can we expect from our most valuable customers?

To get this data we'll need to join the `valuable_customers` table we created to the predicted transaction volumes table `TRANSACTIONS_PRED_predictions`.

In [54]:
trans_y = "TRANSACTIONS_PRED_predictions"

predicted_volume = f"""
SELECT cust.customer_id, trans.target_pred
FROM ({valuable_customers}) cust
INNER JOIN (
    SELECT entity, target_pred
    FROM {dataset_id}.{trans_y}
) trans ON cust.customer_id = trans.entity;
"""

q = client.query(predicted_volume)
rows = q.result()

In [55]:
cust_volume = rows.to_dataframe()
cust_volume

/Users/jamesbriggs/Documents/aurelio/kumo-advocacy/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,customer_id,target_pred
0,2cabdc6101018f8cea44310343769715049befed47caa9...,19.226614
1,77db96923d20d40532eba0020b55cd91eb51358885c2d6...,10.042411
2,062234bcfa5875d71069215348a11f100aa15edd540868...,12.537105
3,2baed3260d6a0c2f23737d09b68d30eff348eb8ec428e0...,15.382269
4,788785852eddb5874f924603105f315d69571b3e5180f3...,10.424101
5,60c8dfc36653461f03d6001b77e7cf6182cf2d71f914c9...,4.758006
6,702ae6f3f3d19e1ca226127f032474aead6247fdd63fe2...,5.925578
7,2111bda116589a94af08443b78cc9dc0944bbf9b3650b9...,13.788486
8,1f8dec83774287b328af151054ffb2e06775ba0551c11b...,7.979001
9,17d6270f6f81ad1f7e5a1cb7ed8edb54bc00d0d5c2cde6...,11.729904


---